In [1]:
"""Test Runner for the Favorite Method"""

import numpy as np
from desdeo.tools import PyomoIpoptSolver
from desdeo.tools.scalarization import add_asf_diff
from desdeo.problem.testproblems.dtlz2_problem import dtlz2

# Import Logic
from desdeo.gdm.favorite_method import (
    IPR_Options, GPRMOptions, ZoomOptions, FavOptions,
    favorite_method, find_candidates, hausdorff_candidates, cluster_points,
    expand_and_generate_candidates
)

# Import Visualization
from visualizations import visualize_selection_2d, visualize_3d_clusters, visualize_expansion


In [2]:
# --- 1. SETUP PROBLEM & DMs ---
dtlz2_problem = dtlz2(30, 3)
n_of_dms = 3

# Generate random reference points and find MPS
reference_points = {}
for i in range(n_of_dms):
    reference_points[f"DM{i+1}"] = {"f_1": np.random.random(), "f_2": np.random.random(), "f_3": np.random.random()}

reference_points = {
    "DM1": {"f_1": 0.0, "f_2": 0.9, "f_3": 0.5},
    "DM2": {"f_1": 0.9, "f_2": 0.1, "f_3": 0.9},
    "DM3": {"f_1": 0.0, "f_2": 0.5, "f_3": 0.9},
    #"DM4": {"f_1": 0.29, "f_2": 0.25, "f_3": 0.20},
}

print("Initial Reference Points:", reference_points)

most_preferred_solutions = {}
for dm in reference_points.keys():
    p, target = add_asf_diff(dtlz2_problem, symbol="asf", reference_point=reference_points[dm])
    solver = PyomoIpoptSolver(p)
    res = solver.solve(target)
    most_preferred_solutions[f"{dm}"] = res.optimal_objectives

Initial Reference Points: {'DM1': {'f_1': 0.0, 'f_2': 0.9, 'f_3': 0.5}, 'DM2': {'f_1': 0.9, 'f_2': 0.1, 'f_3': 0.9}, 'DM3': {'f_1': 0.0, 'f_2': 0.5, 'f_3': 0.9}}


In [3]:
most_preferred_solutions

{'DM1': {'f_1': -1.1596416103593713e-08,
  'f_2': 0.8977817512448849,
  'f_3': 0.44044060567988896},
 'DM2': {'f_1': 0.7973347540605265,
  'f_2': -1.1615413809706364e-08,
  'f_3': 0.6035373144779365},
 'DM3': {'f_1': -4.414719164252631e-09,
  'f_2': 0.44058071330695436,
  'f_3': 0.8977130026138282}}

In [4]:
# --- 2. CONFIGURE OPTIONS ---
ipr_options = IPR_Options(
    most_preferred_solutions=most_preferred_solutions,
    num_initial_reference_points=10000,
    version="box",
)
grpmoptions = GPRMOptions(method_options=ipr_options)
zoomoptions = ZoomOptions(num_steps_remaining=4)

fav_options = FavOptions(
    GPRMoptions=grpmoptions,
    candidate_generation_options="mm",
    zoom_options=zoomoptions,
    original_most_preferred_solutions=most_preferred_solutions,
    votes=None,
    total_n_of_candidates=5 
)

# ITERATION 1: First iteration

In [5]:
# --- 3. RUN ITERATION 1 ---
print("\n--- Running Iteration 1 ---")
fav_results = favorite_method(
    problem=dtlz2_problem,
    options=fav_options,
    results_list=[]
)
print("Iter 1 Complete.")


--- Running Iteration 1 ---
Run 1/100
Run 10/100
Run 20/100
Run 30/100
Run 40/100
Run 50/100
Run 60/100
Run 70/100
Run 80/100
Run 90/100
Run 100/100
[np.float64(2.17931725514612), np.float64(2.0052630917603476), np.float64(2.001141984334704), np.float64(2.0041479713717054), np.float64(1.8759491971368947), np.float64(1.9955799712078193), np.float64(2.2103322318863), np.float64(2.107446324227968), np.float64(1.9480484161903318), np.float64(2.1201713063564855), np.float64(2.2571312054006367), np.float64(1.9950665656705056), np.float64(2.0714161027655575), np.float64(1.908643725304685), np.float64(2.0259378798117096), np.float64(1.9850217140311097), np.float64(1.8856780205416104), np.float64(2.036995862327906), np.float64(2.1410213955648407), np.float64(2.1963335238029824), np.float64(2.057683684884254), np.float64(2.1423331611427994), np.float64(2.1227031999479795), np.float64(1.913198549019983), np.float64(1.9172798930985973), np.float64(2.1538455435643353), np.float64(2.177618538410861

In [6]:
points_matrix, centers_matrix, cluster_labels = find_candidates(fav_results)
all_points = fav_results.GPRMResults.raw_results.evaluated_points
fairs = fav_results.fair_solutions
visualize_3d_clusters(fav_options.GPRMoptions, points_matrix, centers_matrix, cluster_labels, len(fairs), 1)

In [7]:
#most_preferred_solutions

In [8]:
fairs

[FairSolution(objective_values={'f_1': 0.3953713018410788, 'f_2': 0.561893933945593, 'f_3': 0.726606317530777}, fairness_criterion='mm', fairness_value=31.0),
 FairSolution(objective_values={'f_1': 0.6928530082859876, 'f_2': 0.2671659505161223, 'f_3': 0.6697589594726406}, fairness_criterion='avg_hausdorff', fairness_value=0.0),
 FairSolution(objective_values={'f_1': 0.2206174195519807, 'f_2': 0.7876219434406446, 'f_3': 0.5753082898768341}, fairness_criterion='avg_hausdorff', fairness_value=0.0),
 FairSolution(objective_values={'f_1': 0.3154370881242466, 'f_2': 0.2199145856701742, 'f_3': 0.9231126791704315}, fairness_criterion='avg_hausdorff', fairness_value=0.0),
 FairSolution(objective_values={'f_1': 0.6820266608946016, 'f_2': 0.5737366125489879, 'f_3': 0.45350406089667233}, fairness_criterion='avg_hausdorff', fairness_value=0.0)]

In [9]:
# TODO: here we need to interact to get the votes from the DMs. for now, we determine it with the index.
votes = {"DM1": 0, "DM2": 0, "DM3": 1}
winning_idx = 0

In [10]:
# TODO: these would also go somewhere else, some sort of helper function
winning_points = points_matrix[cluster_labels == winning_idx]
winning_center = centers_matrix[winning_idx]

print(f"\nCluster {winning_idx} selected with {len(winning_points)} points.")
print("--- Generating New Candidates via Convex Hull Expansion ---")

fraction_to_keep = 0.8
num_new_points = 1000

new_candidates_k = expand_and_generate_candidates(
    winning_cluster_k=winning_points,
    all_points_k=points_matrix,
    fraction_keep=fraction_to_keep,
    num_new_points=num_new_points
)
print(f"Successfully generated {len(new_candidates_k)} new candidate points.")

# Visualization 3
#visualize_expansion(points_matrix, winning_points, new_candidates_k, winning_center, winning_idx, fraction_to_keep)


Cluster 0 selected with 20 points.
--- Generating New Candidates via Convex Hull Expansion ---
Expanded set: 80 points selected.
Successfully generated 1000 new candidate points.


# ITERATION 2

In [11]:
# --- 6. RUN ITERATION 2 (With Extended Hull) ---
print("\n--- Running Iteration 2 ---")

# Transform numpy array to "Most Preferred Solutions" dict for IPR
next_iter_mps = {}
obj_names = [f"f_{i+1}" for i in range(new_candidates_k.shape[1])]
for i, point in enumerate(new_candidates_k):
    point_dict = {name: val for name, val in zip(obj_names, point)}
    next_iter_mps[f"ext_{i}"] = point_dict

print(next_iter_mps)

# Clone and Update Options
fav_options_2 = fav_options.model_copy(deep=True)
fav_options_2.GPRMoptions.method_options.most_preferred_solutions = next_iter_mps
fav_options_2.GPRMoptions.method_options.version = "convex_hull"  # Use the hull of the new points!
fraction_to_keep = 0.8
fav_options_2.zoom_options.num_steps_remaining = 3


# Voting (DM1 wins previous round)
fav_options_2.votes = votes


--- Running Iteration 2 ---
{'ext_0': {'f_1': np.float64(0.852315424852559), 'f_2': np.float64(1.1900949710039983), 'f_3': np.float64(0.9575896041434427)}, 'ext_1': {'f_1': np.float64(0.914612312319737), 'f_2': np.float64(0.9897410096152578), 'f_3': np.float64(1.0956466780650045)}, 'ext_2': {'f_1': np.float64(0.9628386136500123), 'f_2': np.float64(0.6108360412522834), 'f_3': np.float64(1.4263253450977027)}, 'ext_3': {'f_1': np.float64(1.0349839268769676), 'f_2': np.float64(1.034069236989715), 'f_3': np.float64(0.9309468361333174)}, 'ext_4': {'f_1': np.float64(0.9643270707981649), 'f_2': np.float64(0.6830787331285265), 'f_3': np.float64(1.3525941960733072)}, 'ext_5': {'f_1': np.float64(0.6333504825062631), 'f_2': np.float64(1.154471657203095), 'f_3': np.float64(1.212177860290641)}, 'ext_6': {'f_1': np.float64(0.6638562809116185), 'f_2': np.float64(0.9121781718225431), 'f_3': np.float64(1.423965547265837)}, 'ext_7': {'f_1': np.float64(1.1659952474517312), 'f_2': np.float64(0.85721582622

In [12]:
fav_results_2 = favorite_method(
    problem=dtlz2_problem,
    options=fav_options_2,
    results_list=[fav_results]
)

print("Iter 2 Complete.")
print("Fair Solutions found in Iter 2:", len(fav_results_2.fair_solutions))

[FavResults(FavOptions=FavOptions(GPRMoptions=GPRMOptions(method_options=IPR_Options(num_initial_reference_points=10000, version='box', most_preferred_solutions={'DM1': {'f_1': -1.1596416103593713e-08, 'f_2': 0.8977817512448849, 'f_3': 0.44044060567988896}, 'DM2': {'f_1': 0.7973347540605265, 'f_2': -1.1615413809706364e-08, 'f_3': 0.6035373144779365}, 'DM3': {'f_1': -4.414719164252631e-09, 'f_2': 0.44058071330695436, 'f_3': 0.8977130026138282}}), fake_ideal={'f_1': -1.1596416103593713e-08, 'f_2': -1.1615413809706364e-08, 'f_3': 0.44044060567988896}, fake_nadir={'f_1': 0.7973347540605265, 'f_2': 0.8977817512448849, 'f_3': 0.8977130026138282}, num_points_to_evaluate=100), candidate_generation_options='mm', zoom_options=ZoomOptions(method='nautilus', num_steps_remaining=4), original_most_preferred_solutions={'DM1': {'f_1': -1.1596416103593713e-08, 'f_2': 0.8977817512448849, 'f_3': 0.44044060567988896}, 'DM2': {'f_1': 0.7973347540605265, 'f_2': -1.1615413809706364e-08, 'f_3': 0.603537314477

In [13]:
points_matrix, centers_matrix, cluster_labels = find_candidates(fav_results_2)
all_points = fav_results_2.GPRMResults.raw_results.evaluated_points
fairs = fav_results_2.fair_solutions
visualize_3d_clusters(fav_options_2.GPRMoptions, points_matrix, centers_matrix, cluster_labels, len(fairs), 2)

In [14]:
fairs

[FairSolution(objective_values={'f_1': 0.3953713018410788, 'f_2': 0.561893933945593, 'f_3': 0.726606317530777}, fairness_criterion='mm', fairness_value=31.0),
 FairSolution(objective_values={'f_1': 0.3179015625621187, 'f_2': 0.6174940677176354, 'f_3': 0.7194718012918169}, fairness_criterion='mm', fairness_value=66.0),
 FairSolution(objective_values={'f_1': 0.2919468245322624, 'f_2': 0.22094780194222588, 'f_3': 0.9305638723174392}, fairness_criterion='avg_hausdorff', fairness_value=0.0),
 FairSolution(objective_values={'f_1': 0.5844934559561681, 'f_2': 0.3167911896701724, 'f_3': 0.7470011660578395}, fairness_criterion='avg_hausdorff', fairness_value=0.0),
 FairSolution(objective_values={'f_1': 0.11817005424376902, 'f_2': 0.4774363219356516, 'f_3': 0.870683867300056}, fairness_criterion='avg_hausdorff', fairness_value=0.0)]

In [15]:
# TODO: here we need to interact to get the votes from the DMs. for now, we determine it with the index.
votes = {"DM1": 1, "DM2": 1, "DM3": 4, "DM4": 1}
winning_idx = 1

In [16]:
# TODO: these would also go somewhere else, some sort of helper function
winning_points = points_matrix[cluster_labels == winning_idx]
winning_center = centers_matrix[winning_idx]

print(f"\nCluster {winning_idx} selected with {len(winning_points)} points.")
print("--- Generating New Candidates via Convex Hull Expansion ---")

num_new_points = 1000

new_candidates_k = expand_and_generate_candidates(
    winning_cluster_k=winning_points,
    all_points_k=points_matrix,
    fraction_keep=fraction_to_keep,
    num_new_points=num_new_points
)
print(f"Successfully generated {len(new_candidates_k)} new candidate points.")

# Visualization 3
#visualize_expansion(points_matrix, winning_points, new_candidates_k, winning_center, winning_idx, fraction_to_keep)


Cluster 1 selected with 15 points.
--- Generating New Candidates via Convex Hull Expansion ---
Expanded set: 80 points selected.
Successfully generated 1000 new candidate points.


# ITERATION 3: Further iterations

In [17]:
# --- 6. RUN ITERATION 2 (With Extended Hull) ---
print("\n--- Running Iteration X ---")

# Transform numpy array to "Most Preferred Solutions" dict for IPR
next_iter_mps = {}
obj_names = [f"f_{i+1}" for i in range(new_candidates_k.shape[1])]
for i, point in enumerate(new_candidates_k):
    point_dict = {name: val for name, val in zip(obj_names, point)}
    next_iter_mps[f"ext_{i}"] = point_dict

print(next_iter_mps)

# Clone and Update Options
fav_options_3 = fav_options_2.model_copy(deep=True)
fav_options_3.GPRMoptions.method_options.most_preferred_solutions = next_iter_mps
fav_options_3.GPRMoptions.method_options.version = "convex_hull"  # Use the hull of the new points!
fav_options_3.zoom_options.num_steps_remaining = 2
fraction_to_keep = 0.4

# Voting (DM1 wins previous round)
fav_options_3.votes = votes


--- Running Iteration X ---
{'ext_0': {'f_1': np.float64(1.046750612107394), 'f_2': np.float64(0.9448776154936586), 'f_3': np.float64(1.0083717723989474)}, 'ext_1': {'f_1': np.float64(0.8773322718562887), 'f_2': np.float64(1.011723148195379), 'f_3': np.float64(1.110944579948332)}, 'ext_2': {'f_1': np.float64(0.6707296975197815), 'f_2': np.float64(0.8229843236421789), 'f_3': np.float64(1.5062859788380378)}, 'ext_3': {'f_1': np.float64(0.962025904487909), 'f_2': np.float64(1.0385197760685267), 'f_3': np.float64(0.999454319443564)}, 'ext_4': {'f_1': np.float64(0.6678585602467247), 'f_2': np.float64(1.1278605914638833), 'f_3': np.float64(1.2042808482893914)}, 'ext_5': {'f_1': np.float64(0.5916158464161819), 'f_2': np.float64(0.9568053204663499), 'f_3': np.float64(1.4515788331174666)}, 'ext_6': {'f_1': np.float64(0.8984239280882979), 'f_2': np.float64(0.9418791110598713), 'f_3': np.float64(1.15969696085183)}, 'ext_7': {'f_1': np.float64(0.6083712017068955), 'f_2': np.float64(1.014931153678

In [18]:
fav_results_3 = favorite_method(
    problem=dtlz2_problem,
    options=fav_options_3,
    results_list=[fav_results_2]
)
print("Iter 3 Complete.")
print("Fair Solutions found in Iter 3:", len(fav_results_3.fair_solutions))

[FavResults(FavOptions=FavOptions(GPRMoptions=GPRMOptions(method_options=IPR_Options(num_initial_reference_points=10000, version='convex_hull', most_preferred_solutions={'ext_0': {'f_1': np.float64(0.852315424852559), 'f_2': np.float64(1.1900949710039983), 'f_3': np.float64(0.9575896041434427)}, 'ext_1': {'f_1': np.float64(0.914612312319737), 'f_2': np.float64(0.9897410096152578), 'f_3': np.float64(1.0956466780650045)}, 'ext_2': {'f_1': np.float64(0.9628386136500123), 'f_2': np.float64(0.6108360412522834), 'f_3': np.float64(1.4263253450977027)}, 'ext_3': {'f_1': np.float64(1.0349839268769676), 'f_2': np.float64(1.034069236989715), 'f_3': np.float64(0.9309468361333174)}, 'ext_4': {'f_1': np.float64(0.9643270707981649), 'f_2': np.float64(0.6830787331285265), 'f_3': np.float64(1.3525941960733072)}, 'ext_5': {'f_1': np.float64(0.6333504825062631), 'f_2': np.float64(1.154471657203095), 'f_3': np.float64(1.212177860290641)}, 'ext_6': {'f_1': np.float64(0.6638562809116185), 'f_2': np.float64(

# RUN the voting & visualizations

In [19]:
points_matrix, centers_matrix, cluster_labels = find_candidates(fav_results_3)

all_points = fav_results_3.GPRMResults.raw_results.evaluated_points
fairs = fav_results_3.fair_solutions

visualize_3d_clusters(fav_options_3.GPRMoptions, points_matrix, centers_matrix, cluster_labels, len(fairs), 3)

In [20]:
fairs

[FairSolution(objective_values={'f_1': 0.3179015625621187, 'f_2': 0.6174940677176354, 'f_3': 0.7194718012918169}, fairness_criterion='mm', fairness_value=66.0),
 FairSolution(objective_values={'f_1': 0.30917516677085055, 'f_2': 0.5600359449371669, 'f_3': 0.7686159357120768}, fairness_criterion='mm', fairness_value=75.0),
 FairSolution(objective_values={'f_1': 0.28238294974068034, 'f_2': 0.22575662596077764, 'f_3': 0.9323592738481009}, fairness_criterion='avg_hausdorff', fairness_value=0.0),
 FairSolution(objective_values={'f_1': 0.5438505061410047, 'f_2': 0.3108904281471239, 'f_3': 0.7794701845848057}, fairness_criterion='avg_hausdorff', fairness_value=0.0),
 FairSolution(objective_values={'f_1': 0.0785519975424131, 'f_2': 0.19013585720307025, 'f_3': 0.9786102081460986}, fairness_criterion='avg_hausdorff', fairness_value=0.0)]

In [21]:
# TODO: here we need to interact to get the votes from the DMs. for now, we determine it with the index.
votes = {"DM1":2, "DM2": 2, "DM3": 2, "DM4": 0}
winning_idx = 2

In [22]:
# TODO: these would also go somewhere else, some sort of helper function
winning_points = points_matrix[cluster_labels == winning_idx]
winning_center = centers_matrix[winning_idx]

print(f"\nCluster {winning_idx} selected with {len(winning_points)} points.")
print("--- Generating New Candidates via Convex Hull Expansion ---")

num_new_points = 1000

new_candidates_k = expand_and_generate_candidates(
    winning_cluster_k=winning_points,
    all_points_k=points_matrix,
    fraction_keep=fraction_to_keep,
    num_new_points=num_new_points
)
print(f"Successfully generated {len(new_candidates_k)} new candidate points.")

# Visualization 3
#visualize_expansion(points_matrix, winning_points, new_candidates_k, winning_center, winning_idx, fraction_to_keep)


Cluster 2 selected with 27 points.
--- Generating New Candidates via Convex Hull Expansion ---
Expanded set: 40 points selected.
Successfully generated 1000 new candidate points.


In [23]:
# --- 6. RUN ITERATION 2 (With Extended Hull) ---
print("\n--- Running Iteration X ---")

# Transform numpy array to "Most Preferred Solutions" dict for IPR
next_iter_mps = {}
obj_names = [f"f_{i+1}" for i in range(new_candidates_k.shape[1])]
for i, point in enumerate(new_candidates_k):
    point_dict = {name: val for name, val in zip(obj_names, point)}
    next_iter_mps[f"ext_{i}"] = point_dict

print(next_iter_mps)

# Clone and Update Options
fav_options_4 = fav_options_3.model_copy(deep=True)
fav_options_4.GPRMoptions.method_options.most_preferred_solutions = next_iter_mps
fav_options_4.GPRMoptions.method_options.version = "convex_hull"  # Use the hull of the new points!
fraction_to_keep = 0.1
fav_options_4.zoom_options.num_steps_remaining = 1


# Voting (DM1 wins previous round)
fav_options_4.votes = votes


--- Running Iteration X ---
{'ext_0': {'f_1': np.float64(0.7836713743143581), 'f_2': np.float64(0.7124755287510748), 'f_3': np.float64(1.5038530969345656)}, 'ext_1': {'f_1': np.float64(0.9653459162661405), 'f_2': np.float64(0.6108933075945188), 'f_3': np.float64(1.4237607761393392)}, 'ext_2': {'f_1': np.float64(0.7069255206111725), 'f_2': np.float64(0.897662535424084), 'f_3': np.float64(1.395411943964742)}, 'ext_3': {'f_1': np.float64(0.9481310499097985), 'f_2': np.float64(0.6825141149570143), 'f_3': np.float64(1.3693548351331861)}, 'ext_4': {'f_1': np.float64(0.8284785577387316), 'f_2': np.float64(0.734074139337528), 'f_3': np.float64(1.4374473029237387)}, 'ext_5': {'f_1': np.float64(0.7515301413649119), 'f_2': np.float64(0.8851827210389508), 'f_3': np.float64(1.363287137596136)}, 'ext_6': {'f_1': np.float64(0.7822143133650824), 'f_2': np.float64(0.8108403622310062), 'f_3': np.float64(1.4069453244039098)}, 'ext_7': {'f_1': np.float64(0.6859916284670841), 'f_2': np.float64(0.827930789

In [24]:
fav_results_4 = favorite_method(
    problem=dtlz2_problem,
    options=fav_options_4,
    results_list=[
        fav_results_3,
    ]
)
print("Iter 4 Complete.")
print("Fair Solutions found in Iter 4:", len(fav_results_4.fair_solutions))

[FavResults(FavOptions=FavOptions(GPRMoptions=GPRMOptions(method_options=IPR_Options(num_initial_reference_points=10000, version='convex_hull', most_preferred_solutions={'ext_0': {'f_1': np.float64(1.046750612107394), 'f_2': np.float64(0.9448776154936586), 'f_3': np.float64(1.0083717723989474)}, 'ext_1': {'f_1': np.float64(0.8773322718562887), 'f_2': np.float64(1.011723148195379), 'f_3': np.float64(1.110944579948332)}, 'ext_2': {'f_1': np.float64(0.6707296975197815), 'f_2': np.float64(0.8229843236421789), 'f_3': np.float64(1.5062859788380378)}, 'ext_3': {'f_1': np.float64(0.962025904487909), 'f_2': np.float64(1.0385197760685267), 'f_3': np.float64(0.999454319443564)}, 'ext_4': {'f_1': np.float64(0.6678585602467247), 'f_2': np.float64(1.1278605914638833), 'f_3': np.float64(1.2042808482893914)}, 'ext_5': {'f_1': np.float64(0.5916158464161819), 'f_2': np.float64(0.9568053204663499), 'f_3': np.float64(1.4515788331174666)}, 'ext_6': {'f_1': np.float64(0.8984239280882979), 'f_2': np.float64(

In [25]:
points_matrix, centers_matrix, cluster_labels = find_candidates(fav_results_4)

all_points = fav_results_4.GPRMResults.raw_results.evaluated_points
fairs = fav_results_4.fair_solutions

visualize_3d_clusters(fav_options_4.GPRMoptions, points_matrix, centers_matrix, cluster_labels, len(fairs), 4)

In [26]:
fairs

[FairSolution(objective_values={'f_1': 0.28238294974068034, 'f_2': 0.22575662596077764, 'f_3': 0.9323592738481009}, fairness_criterion='avg_hausdorff', fairness_value=0.0),
 FairSolution(objective_values={'f_1': 0.39714469058961444, 'f_2': 0.3388315403905765, 'f_3': 0.8529180980451925}, fairness_criterion='mm', fairness_value=8.0),
 FairSolution(objective_values={'f_1': 0.4594656401814896, 'f_2': 0.4988435284808234, 'f_3': 0.734878533898913}, fairness_criterion='avg_hausdorff', fairness_value=0.0),
 FairSolution(objective_values={'f_1': 0.5727971189259276, 'f_2': 0.330992833298889, 'f_3': 0.7498981296515752}, fairness_criterion='avg_hausdorff', fairness_value=0.0),
 FairSolution(objective_values={'f_1': 0.3260565331180251, 'f_2': 0.5255552585724459, 'f_3': 0.7857981976295846}, fairness_criterion='avg_hausdorff', fairness_value=0.0)]

In [27]:
# FINAL STEP GO TO VOTING
votes = {"DM1": 0, "DM2": 0, "DM3": 0, "DM4": 0}
winning_idx = 0

In [28]:
# TODO: these would also go somewhere else, some sort of helper function
winning_points = points_matrix[cluster_labels == winning_idx]
winning_center = centers_matrix[winning_idx]

print(f"\nCluster {winning_idx} selected with {len(winning_points)} points.")
print("--- Generating New Candidates via Convex Hull Expansion ---")

num_new_points = 1000

new_candidates_k = expand_and_generate_candidates(
    winning_cluster_k=winning_points,
    all_points_k=points_matrix,
    fraction_keep=fraction_to_keep,
    num_new_points=num_new_points
)
print(f"Successfully generated {len(new_candidates_k)} new candidate points.")

# Visualization 3
#visualize_expansion(points_matrix, winning_points, new_candidates_k, winning_center, winning_idx, fraction_to_keep)


Cluster 0 selected with 0 points.
--- Generating New Candidates via Convex Hull Expansion ---


IndexError: index 0 is out of bounds for axis 0 with size 0